# pp EEC AUT predictions for Fig. 4

This notebook is the pp-only prediction driver for the Fig. 4 data in `Data/pp EEC AUT`.
It uses `mu = pT`, generates/loads the JAMDiFF grid as `dict_raw_DiFF_pp.pkl`, and evaluates the eta-only bin integration functions in `Processes/pp/pp_AUT.jl`.


In [ ]:
from julia.api import Julia
from julia import Main

import os
import re
import sys
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.std import tqdm

sys.path.insert(0, str(Path("../JAMDiFF_library").resolve()))
from get_JAM_DiFF import get_JAM_DiFF_dict


## Settings

In [ ]:
fit_name = "Default"

# If False, the notebook still generates the pp grid when the _pp pickle does not exist.
if_generate_grids = False

N_replicas = 20
N_loaded_replicas = N_replicas
N_cores = 16

# Fig. 4 is evaluated with fixed theta at the bin center and integrated over eta only.
use_eta_only = True

fig4_data_dir = Path("../Data/pp EEC AUT")
fitting_root = "../"


## Load Fig. 4 Data

In [ ]:
fig4_files = sorted(fig4_data_dir.glob("figure4_pad*_EEC.csv"))
if not fig4_files:
    raise FileNotFoundError(f"No figure4_pad*_EEC.csv files found in {fig4_data_dir}")

frames = []
for path in fig4_files:
    df = pd.read_csv(path)
    match = re.search(r"pad(\d+)", path.name)
    df["pad"] = int(match.group(1)) if match else len(frames) + 1
    df["source_file"] = path.name
    frames.append(df)

fig4_df = pd.concat(frames, ignore_index=True)
fig4_df["theta"] = 0.5 * (fig4_df["θ_lo"].to_numpy(float) + fig4_df["θ_hi"].to_numpy(float))
fig4_df = fig4_df.sort_values(["pad", "theta", "pT"]).reset_index(drop=True)

mu_array = np.sort(fig4_df["pT"].unique().astype(float))
print(f"Loaded {len(fig4_df)} Fig. 4 points from {len(fig4_files)} files")
print("mu = pT grid:", mu_array)
display(fig4_df.head())


## Initialize Julia

In [ ]:
Main.eval('using Distributed')
existing_workers = list(Main.eval('workers()'))
if existing_workers:
    Main.eval('rmprocs(workers())')
Main.eval(f'addprocs({N_cores})')

def include(name):
    path = os.path.join(fitting_root, name)
    Main.eval(f'@everywhere include(raw"{path}")')

card_name = fit_name.removesuffix(".jl")
include(f"Cards/{card_name}.jl")

include("Collinear_PDF/pdf.jl")
include("Core/constants.jl")
include("Core/strong coupling.jl")
include("Numerical/FastGK.jl")
include("Numerical/MultiGKTable.jl")
include("DiFF_EEC/DiFF_EEC.jl")
include("Processes/pp/hard.jl")
include("Processes/pp/pp_AUT.jl")

wdir = Main.eval("wdir")
scan_grid_name = str(Main.eval("scan_grid_name"))
dict_raw_DiFF_base_name = str(Main.eval("dict_raw_DiFF_name"))

scan_coeffs_path = Path(f"{scan_grid_name}.xlsx")
dict_raw_DiFF_path = Path("../Grids") / f"{dict_raw_DiFF_base_name}_pp.pkl"
dict_raw_DiFF_path.parent.mkdir(parents=True, exist_ok=True)

plot_data_dir = Path("Plot_Data") / fit_name
figures_dir = Path("Plot_Data") / "Figures"
plot_data_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

print("wdir:", wdir)
print("scan coeffs:", scan_coeffs_path)
print("JAMDiFF pp grid:", dict_raw_DiFF_path)


## Generate Or Load JAMDiFF Grid

In [ ]:
z_array = np.linspace(0.19, 0.99, 20)
Mh_array = np.linspace(0.28, 2.05, 50)
kinds = ["D1", "H1a"]

if if_generate_grids or not dict_raw_DiFF_path.exists():
    print(f"Generating {dict_raw_DiFF_path.name} for mu = pT values")
    dict_raw_DiFF_full = get_JAM_DiFF_dict(
        z_array=z_array,
        Mh_array=Mh_array,
        mu_array=mu_array,
        kinds=kinds,
        wdir=wdir,
    )
    with dict_raw_DiFF_path.open("wb") as f:
        pickle.dump(dict_raw_DiFF_full, f, protocol=pickle.HIGHEST_PROTOCOL)
else:
    with dict_raw_DiFF_path.open("rb") as f:
        dict_raw_DiFF_full = pickle.load(f)

keys = [k for k in dict_raw_DiFF_full.keys() if isinstance(k, tuple)]
all_replica_ids = np.asarray(sorted({k[2] for k in keys if k[2] > 0}), dtype=int)
N_total = len(all_replica_ids)
print(f"Available JAMDiFF replicas: {N_total}")


## Construct JAMDiFF Interpolators

In [ ]:
n_selected = min(int(N_loaded_replicas), N_total)
selected_positive_replica_ids = np.asarray(
    np.random.choice(all_replica_ids, size=n_selected, replace=False),
    dtype=int,
)
selected_positive_replica_ids = np.sort(selected_positive_replica_ids)
replica_ids = np.sort(np.concatenate(([0], selected_positive_replica_ids)))
selected_replica_set = set(replica_ids.tolist())

# Reduce the payload before broadcasting it to workers.
dict_raw_DiFF = {
    k: v for k, v in dict_raw_DiFF_full.items()
    if (not isinstance(k, tuple)) or (k[2] in selected_replica_set)
}
keys = [k for k in dict_raw_DiFF.keys() if isinstance(k, tuple)]
selected_keys = list(keys)

Main.dict_raw_DiFF = dict_raw_DiFF
Main.eval('''let payload = dict_raw_DiFF
    @everywhere global dict_raw_DiFF = $payload
end''')

Main.selected_keys = selected_keys
Main.eval('''let payload = selected_keys
    @everywhere global selected_keys = $payload
end''')

include("Grids/initialization.jl")
del dict_raw_DiFF_full

print(f"Loaded {len(selected_positive_replica_ids)} replica(s) plus replica 0 from {N_total} available")


## CT18 And Scan Helpers

In [ ]:
try:
    import lhapdf
except ImportError:
    lhapdf = None

ct18_pdfset_name = str(Main.eval('pdf_dict_array[1]["pdfset_name"]'))

def _default_ct18_positive_member_ids():
    if lhapdf is not None:
        try:
            return np.arange(1, len(lhapdf.mkPDFs(ct18_pdfset_name)), dtype=int)
        except Exception:
            pass
    if ct18_pdfset_name == "CT18NLO_mc":
        return np.arange(1, 101, dtype=int)
    raise RuntimeError(f"Could not determine positive PDF members for {ct18_pdfset_name}")

ct18_positive_member_ids = _default_ct18_positive_member_ids()

def _sample_ct18_members(n):
    n = int(n)
    if n <= 0:
        return np.empty(0, dtype=int)
    return np.asarray(np.random.choice(ct18_positive_member_ids, size=n, replace=True), dtype=int)

def _read_scan_coeffs(path):
    path = Path(path)
    try:
        df = pd.read_excel(path)
        if {"a", "b"}.issubset(df.columns):
            df = df[["a", "b"]]
        else:
            raise ValueError
    except Exception:
        df = pd.read_excel(path, header=None).iloc[:, :2]
        df.columns = ["a", "b"]
    df = df.apply(pd.to_numeric, errors="coerce").dropna(how="any").reset_index(drop=True)
    if df.empty:
        raise ValueError(f"No valid (a, b) pairs found in {path}")
    return df[["a", "b"]].to_numpy(dtype=float)

def _sample_coeff_assignments(coeff_grid, n_assignments):
    coeff_grid = np.asarray(coeff_grid, dtype=float)
    n_assignments = int(n_assignments)
    if n_assignments <= 0:
        return np.empty((0, 2), dtype=float), np.empty(0, dtype=int)
    idx = np.asarray(np.random.choice(np.arange(len(coeff_grid)), size=n_assignments, replace=True), dtype=int)
    return np.asarray(coeff_grid[idx], dtype=float), idx


## Replica And Progress Helpers

In [ ]:
def _normalize_replica_ids(replica_ids=None):
    if replica_ids is None:
        return np.asarray(selected_positive_replica_ids, dtype=int)
    if isinstance(replica_ids, str):
        if replica_ids.lower() in {"loaded", "default"}:
            return np.asarray(selected_positive_replica_ids, dtype=int)
        raise ValueError(f"Unsupported replica selection: {replica_ids}")
    replica_ids = np.asarray(replica_ids, dtype=int)
    unknown = np.setdiff1d(replica_ids, selected_positive_replica_ids)
    if unknown.size:
        raise ValueError(f"Requested replica ids were not loaded: {unknown.tolist()}")
    return np.unique(replica_ids)

def _sample_replica_assignments(replica_pool, n_assignments=None):
    replica_pool = np.asarray(replica_pool, dtype=int)
    if n_assignments is None:
        n_assignments = N_replicas
    n_assignments = int(n_assignments)
    if n_assignments <= 0:
        return np.empty(0, dtype=int)
    return np.asarray(np.random.choice(replica_pool, size=n_assignments, replace=True), dtype=int)


## Julia Prediction Helper

In [ ]:
Main.eval('''
@everywhere begin
function _pp_pmap_batch_size(n::Integer)
    return max(1, min(64, cld(n, max(nworkers(), 1) * 4)))
end

function _pp_quiet_set_lhapdf(i_set::Int64, name::String, i_member::Int64)
    redirect_stdout(devnull) do
        redirect_stderr(devnull) do
            set_lhapdf(i_set, name, i_member)
        end
    end
    return nothing
end

function pp_EEC_AUT_bin_eta_pmap_one_replica_with_coeffs_and_pdfmember(; pT_array::AbstractVector, theta_array::AbstractVector, eta_lo_array::AbstractVector, eta_hi_array::AbstractVector, sqrts_array::AbstractVector, rep::Integer, pdf_member::Integer, a::Real, b::Real)
    n_points = length(pT_array)
    length(theta_array) == n_points || throw(ArgumentError("theta_array must have the same length as pT_array"))
    length(eta_lo_array) == n_points || throw(ArgumentError("eta_lo_array must have the same length as pT_array"))
    length(eta_hi_array) == n_points || throw(ArgumentError("eta_hi_array must have the same length as pT_array"))
    length(sqrts_array) == n_points || throw(ArgumentError("sqrts_array must have the same length as pT_array"))

    rep_i = Int(rep)
    pdf_member_i = Int(pdf_member)
    a_i = Float64(a)
    b_i = Float64(b)
    point_args = collect(zip(Float64.(pT_array), Float64.(theta_array), Float64.(eta_lo_array), Float64.(eta_hi_array), Float64.(sqrts_array)))

    predictions = nothing
    t = @elapsed begin
        predictions = pmap(point_args; batch_size = _pp_pmap_batch_size(length(point_args))) do (pT, theta, eta_lo, eta_hi, sqrts)
            Core.eval(Main, :(a_D1 = $(a_i)))
            Core.eval(Main, :(b_D1 = $(b_i)))
            Core.eval(Main, :(a_H1a = $(a_i)))
            Core.eval(Main, :(b_H1a = $(b_i)))
            _pp_quiet_set_lhapdf(0, pdf_dict_array[1]["pdfset_name"], pdf_member_i)
            pp_EEC_AUT_bin_η(pT = pT, θ = theta, η_lo = eta_lo, η_hi = eta_hi, sqrts = sqrts, rep = rep_i)
        end
    end

    return predictions, t
end
end
''')


## Fig. 4 Prediction Helpers

In [ ]:
def _fig4_arrays(df):
    return {
        "pT_array": df["pT"].to_numpy(float),
        "theta_array": df["theta"].to_numpy(float),
        "eta_lo_array": df["η_lo"].to_numpy(float),
        "eta_hi_array": df["η_hi"].to_numpy(float),
        "sqrts_array": df["sqrts"].to_numpy(float),
    }

def _run_pp_one_replica(data_df, rep, pdf_member, coeff):
    arr = _fig4_arrays(data_df)
    coeff = np.asarray(coeff, dtype=float)
    values, elapsed = Main.pp_EEC_AUT_bin_eta_pmap_one_replica_with_coeffs_and_pdfmember(
        **arr,
        rep=int(rep),
        pdf_member=int(pdf_member),
        a=float(coeff[0]),
        b=float(coeff[1]),
    )
    return np.asarray(values, dtype=float), float(elapsed)


In [ ]:
def generate_pp_fig4_predictions(*, data_df, coeffs_path, replica_ids="loaded", save=True):
    data_df = data_df.reset_index(drop=True).copy()
    n_points = len(data_df)
    coeff_grid = _read_scan_coeffs(coeffs_path)
    replica_pool = _normalize_replica_ids(replica_ids)

    sampled_replica_ids = _sample_replica_assignments(replica_pool, N_replicas)
    pdf_member_ids = _sample_ct18_members(N_replicas)
    sampled_coeff_grid, sampled_coeff_indices = _sample_coeff_assignments(coeff_grid, N_replicas)

    samples = np.empty((N_replicas, n_points), dtype=float)
    replica_elapsed = np.empty(N_replicas, dtype=float)

    with tqdm(
        total=N_replicas,
        desc="pp Fig. 4 replicas",
        unit="replica",
        file=sys.stdout,
        dynamic_ncols=False,
        ascii=True,
        mininterval=0.0,
        miniters=1,
        leave=True,
    ) as pbar:
        pbar.refresh()
        for i, (rep_i, pdf_member_i, coeff_i) in enumerate(zip(sampled_replica_ids, pdf_member_ids, sampled_coeff_grid)):
            pbar.set_postfix_str(f"running {i + 1}/{N_replicas}", refresh=True)
            values, elapsed = _run_pp_one_replica(
                data_df,
                rep=rep_i,
                pdf_member=pdf_member_i,
                coeff=coeff_i,
            )
            samples[i, :] = values
            replica_elapsed[i] = elapsed
            pbar.update(1)
            pbar.set_postfix(
                last=f"{elapsed:.1f}s",
                mean=f"{replica_elapsed[:i + 1].mean():.1f}s",
                refresh=True,
            )

    print(
        "Replica wall time: "
        f"mean={replica_elapsed.mean():.1f}s, "
        f"median={np.median(replica_elapsed):.1f}s, "
        f"min={replica_elapsed.min():.1f}s, "
        f"max={replica_elapsed.max():.1f}s"
    )

    p16, p50, p84 = np.percentile(samples, [16, 50, 84], axis=0)
    pred_df = data_df.copy()
    pred_df["theta"] = pred_df["theta"].astype(float)
    pred_df["prediction_p16"] = p16
    pred_df["prediction_p50"] = p50
    pred_df["prediction_p84"] = p84
    pred_df["n_prediction_members"] = samples.shape[0]

    result = {
        "process": "pp_EEC_AUT_fig4",
        "fit_name": fit_name,
        "card_name": card_name,
        "wdir": wdir,
        "data_files": [str(p) for p in fig4_files],
        "use_eta_only": use_eta_only,
        "theta_definition": "0.5 * (theta_lo + theta_hi)",
        "mu_definition": "mu = pT",
        "dict_raw_DiFF_path": str(dict_raw_DiFF_path.resolve()),
        "scan_coeffs_path": str(Path(coeffs_path).resolve()),
        "coeff_grid": coeff_grid,
        "selected_replica_ids": np.asarray(selected_positive_replica_ids, dtype=int),
        "sampled_replica_ids": sampled_replica_ids,
        "pdfset_name": ct18_pdfset_name,
        "pdf_member_ids": pdf_member_ids,
        "sampled_coeff_grid": sampled_coeff_grid,
        "sampled_coeff_indices": sampled_coeff_indices,
        "replica_elapsed": replica_elapsed,
        "data": data_df,
        "samples": samples,
        "predictions": pred_df,
    }

    if save:
        pkl_path = plot_data_dir / "pp_fig4_predictions.pkl"
        csv_path = plot_data_dir / "pp_fig4_predictions.csv"
        with pkl_path.open("wb") as f:
            pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)
        pred_df.to_csv(csv_path, index=False)
        print(f"Saved {pkl_path}")
        print(f"Saved {csv_path}")

    return result, pred_df


## Run Fig. 4 Predictions

In [ ]:
pp_fig4_results, pp_fig4_predictions = generate_pp_fig4_predictions(
    data_df=fig4_df,
    coeffs_path=scan_coeffs_path,
    replica_ids="loaded",
    save=True,
)

display(pp_fig4_predictions.head())


## Plot Fig. 4 Prediction

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6.5), sharex=True, sharey=True)
axes = axes.ravel()

for ax, pad in zip(axes, sorted(pp_fig4_predictions["pad"].unique())):
    df = pp_fig4_predictions[pp_fig4_predictions["pad"] == pad].sort_values("theta")
    theta = df["theta"].to_numpy(float)
    y = df["AUT"].to_numpy(float)
    yerr = np.sqrt(df["error_stat"].to_numpy(float)**2 + df["error_sys"].to_numpy(float)**2)

    ax.errorbar(theta, y, yerr=yerr, fmt="o", color="black", ms=4, capsize=2, label="STAR data")
    ax.plot(theta, df["prediction_p50"], color="#1f77b4", lw=1.8, label="prediction median")
    ax.fill_between(theta, df["prediction_p16"], df["prediction_p84"], color="#1f77b4", alpha=0.25, label="68% band")
    ax.axhline(0.0, color="0.75", lw=0.8)
    ax.set_title(f"pad {pad}, pT = {df['pT'].iloc[0]:g} GeV")
    ax.grid(alpha=0.25)

for ax in axes[3:]:
    ax.set_xlabel(r"$\theta$ [rad]")
for ax in axes[::3]:
    ax.set_ylabel(r"$A_{UT}$")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
fig.suptitle("pp EEC AUT Fig. 4 prediction", y=1.02)
fig.tight_layout()

fig_path_pdf = figures_dir / "pp_fig4_prediction.pdf"
fig_path_png = figures_dir / "pp_fig4_prediction.png"
fig.savefig(fig_path_pdf, bbox_inches="tight")
fig.savefig(fig_path_png, dpi=200, bbox_inches="tight")
print(f"Saved {fig_path_pdf}")
print(f"Saved {fig_path_png}")
plt.show()
